In [44]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [45]:
base_dir = '/data/aman_singh/acuuracy_check'
os.chdir(base_dir)

In [46]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [47]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper functions

In [48]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [49]:
blinkit_forecast_apr = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Aug 2026_to_Nov 2026.csv')

blinkit_forecast_apr.rename(columns = {'item_id':'item_code', 'Sep - forecast':'forecast_quantity'}, inplace = True)
blinkit_forecast_apr.drop(columns = ['Aug - forecast', 'Oct - forecast', 'Nov - forecast'],inplace = True)
blinkit_forecast_apr['date'] = '2026-09-30'
blinkit_forecast_apr['date'] = pd.to_datetime(blinkit_forecast_apr['date'])
blinkit_forecast_apr

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,2960,Patna P1 - Feeder Warehouse,10113503,Saffola Gold Corn & Rice Bran Refined Blended ...,1385,Marico Ltd.,5271.0,Marico Ltd,8426,2026-09-30
1,2960,Patna P1 - Feeder Warehouse,10004412,Saffola Classic-Masala Oats(Pack)38 gm - Rs 20.0,1385,Marico Ltd.,5271.0,Marico Ltd,6738,2026-09-30
2,2960,Patna P1 - Feeder Warehouse,10138724,Saffola Crunchiez Ragi (Munchiez) Chips Masala...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-09-30
3,2960,Patna P1 - Feeder Warehouse,10224609,Parachute Advansed Gold Keratin Coconut Hair O...,1385,Marico Ltd.,5271.0,Marico Ltd,113,2026-09-30
4,2960,Patna P1 - Feeder Warehouse,10151975,Saffola Spicy Mexicana Masala Flavoured Oats(P...,1385,Marico Ltd.,5271.0,Marico Ltd,7,2026-09-30
...,...,...,...,...,...,...,...,...,...,...
5517,5096,Faridabad - Feeder Warehouse,10273411,Livon Keratin Damage Repair Shampoo(PET Bottle...,1385,Marico Ltd.,5271.0,Marico Ltd,177,2026-09-30
5518,5096,Faridabad - Feeder Warehouse,10189066,Parachute Advansed Gentle Baby Wash for New Bo...,1385,Marico Ltd.,5271.0,Marico Ltd,150,2026-09-30
5519,5096,Faridabad - Feeder Warehouse,10256289,Parachute Advansed Radiant Shine Hair Serum(Bo...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-09-30
5520,5096,Faridabad - Feeder Warehouse,10029462,Bio-Oil Skin Care Body Oil(Pack)125 ml - Rs 975,1385,Marico Ltd.,5271.0,Marico Ltd,430,2026-09-30


In [50]:
# blinkit_forecast_may = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_May 2026_to_Aug 2026.csv')

# blinkit_forecast_may.rename(columns = {'item_id':'item_code', 'May - forecast':'forecast_quantity'}, inplace = True)
# blinkit_forecast_may.drop(columns = ['Aug - forecast', 'Jun - forecast', 'Jul - forecast'],inplace = True)
# blinkit_forecast_may['date'] = '2026-05-31'
# blinkit_forecast_may['date'] = pd.to_datetime(blinkit_forecast_may['date'])
# blinkit_forecast_may

In [51]:
# blinkit_forecast_june = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Jun 2026_to_Sep 2026.csv')

# blinkit_forecast_june.rename(columns = {'item_id':'item_code', 'Jun - forecast':'forecast_quantity'}, inplace = True)
# blinkit_forecast_june.drop(columns = ['Aug - forecast', 'Sep - forecast', 'Jul - forecast'],inplace = True)
# blinkit_forecast_june['date'] = '2026-06-30'
# blinkit_forecast_june['date'] = pd.to_datetime(blinkit_forecast_june['date'])
# blinkit_forecast_june

In [52]:
#blinkit_unpivoted = pd.concat([blinkit_forecast_apr,blinkit_forecast_may,blinkit_forecast_june])
blinkit_unpivoted = blinkit_forecast_apr.copy()
blinkit_unpivoted['chain_name'] = 'Blinkit'
blinkit_unpivoted

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date,chain_name
0,2960,Patna P1 - Feeder Warehouse,10113503,Saffola Gold Corn & Rice Bran Refined Blended ...,1385,Marico Ltd.,5271.0,Marico Ltd,8426,2026-09-30,Blinkit
1,2960,Patna P1 - Feeder Warehouse,10004412,Saffola Classic-Masala Oats(Pack)38 gm - Rs 20.0,1385,Marico Ltd.,5271.0,Marico Ltd,6738,2026-09-30,Blinkit
2,2960,Patna P1 - Feeder Warehouse,10138724,Saffola Crunchiez Ragi (Munchiez) Chips Masala...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-09-30,Blinkit
3,2960,Patna P1 - Feeder Warehouse,10224609,Parachute Advansed Gold Keratin Coconut Hair O...,1385,Marico Ltd.,5271.0,Marico Ltd,113,2026-09-30,Blinkit
4,2960,Patna P1 - Feeder Warehouse,10151975,Saffola Spicy Mexicana Masala Flavoured Oats(P...,1385,Marico Ltd.,5271.0,Marico Ltd,7,2026-09-30,Blinkit
...,...,...,...,...,...,...,...,...,...,...,...
5517,5096,Faridabad - Feeder Warehouse,10273411,Livon Keratin Damage Repair Shampoo(PET Bottle...,1385,Marico Ltd.,5271.0,Marico Ltd,177,2026-09-30,Blinkit
5518,5096,Faridabad - Feeder Warehouse,10189066,Parachute Advansed Gentle Baby Wash for New Bo...,1385,Marico Ltd.,5271.0,Marico Ltd,150,2026-09-30,Blinkit
5519,5096,Faridabad - Feeder Warehouse,10256289,Parachute Advansed Radiant Shine Hair Serum(Bo...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-09-30,Blinkit
5520,5096,Faridabad - Feeder Warehouse,10029462,Bio-Oil Skin Care Body Oil(Pack)125 ml - Rs 975,1385,Marico Ltd.,5271.0,Marico Ltd,430,2026-09-30,Blinkit


In [53]:
swiggy_forecast_apr = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_aug.xlsx')
swiggy_forecast_apr.columns = swiggy_forecast_apr.columns.str.lower()
swiggy_forecast_apr.rename(columns = {'wh_name':'facility_name', 'sep_buy_qty':'forecast_quantity'}, inplace = True)
swiggy_forecast_apr.drop(columns = ['aug_buy_qty', 'oct_buy_qty'],inplace = True)
swiggy_forecast_apr['date'] = '2026-09-30'
swiggy_forecast_apr['date'] = pd.to_datetime(swiggy_forecast_apr['date'])
swiggy_forecast_apr

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date
0,53164,Just Herbs Multicolor Clean Girl Aesthetic Nai...,MARICO LIMITED,Just Herbs,CENTRAL GOA,GOA IM1,0,2026-09-30
1,53164,Just Herbs Multicolor Clean Girl Aesthetic Nai...,MARICO LIMITED,Just Herbs,MUMBAI,MUM IM3,0,2026-09-30
2,54266,"Saffola Masala Oats Spicy Mexicana, Healthy & ...",MARICO LIMITED,Saffola,GURGAON,DLHY GGNFC5,30,2026-09-30
3,54266,"Saffola Masala Oats Spicy Mexicana, Healthy & ...",MARICO LIMITED,Saffola,VIZAG,VIZ IM1,30,2026-09-30
4,55794,Parachute Advansed Baby Massage Oil (Virgin Co...,MARICO LIMITED,Parachute Advansed,GURGAON,DLHY GGNFC5,0,2026-09-30
...,...,...,...,...,...,...,...,...
10476,990631,"Beardo Beard & Hair Growth Oil, Natural Hair O...",MARICO LIMITED,Beardo,COIMBATORE,CBE ECOM,72,2026-09-30
10477,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,DELHI,DLHY GGNFC9,0,2026-09-30
10478,995855,Beardo UltraGlow Body Wash for Men Moisturizes...,MARICO LIMITED,Beardo,JAIPUR,JAI IM1,0,2026-09-30
10479,998784,Beardo LSD Lust Seduction Desire Eau De Parfum,MARICO LIMITED,Beardo,BANGALORE,BLR IM1,0,2026-09-30


In [54]:
# swiggy_forecast_may = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_may.xlsx')
# swiggy_forecast_may.columns = swiggy_forecast_may.columns.str.lower()
# swiggy_forecast_may.rename(columns = {'wh_name':'facility_name', 'may_buy_qty':'forecast_quantity'}, inplace = True)
# swiggy_forecast_may.drop(columns = ['jul_buy_qty', 'jun_buy_qty'],inplace = True)
# swiggy_forecast_may['date'] = '2026-05-31'
# swiggy_forecast_may['date'] = pd.to_datetime(swiggy_forecast_may['date'])
# swiggy_forecast_may

In [55]:
# swiggy_forecast_jun = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_june.xlsx')
# swiggy_forecast_jun.columns = swiggy_forecast_jun.columns.str.lower()
# swiggy_forecast_jun.rename(columns = {'wh_name':'facility_name', 'jun_buy_qty':'forecast_quantity'}, inplace = True)
# swiggy_forecast_jun.drop(columns = ['jul_buy_qty', 'aug_buy_qty'],inplace = True)
# swiggy_forecast_jun['date'] = '2026-06-30'
# swiggy_forecast_jun['date'] = pd.to_datetime(swiggy_forecast_jun['date'])
# swiggy_forecast_jun

In [56]:
# Swiggy_unpivoted = pd.concat([swiggy_forecast_apr,swiggy_forecast_may,swiggy_forecast_jun])
Swiggy_unpivoted = swiggy_forecast_apr.copy()
Swiggy_unpivoted['chain_name'] = 'Swiggy'
Swiggy_unpivoted

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date,chain_name
0,53164,Just Herbs Multicolor Clean Girl Aesthetic Nai...,MARICO LIMITED,Just Herbs,CENTRAL GOA,GOA IM1,0,2026-09-30,Swiggy
1,53164,Just Herbs Multicolor Clean Girl Aesthetic Nai...,MARICO LIMITED,Just Herbs,MUMBAI,MUM IM3,0,2026-09-30,Swiggy
2,54266,"Saffola Masala Oats Spicy Mexicana, Healthy & ...",MARICO LIMITED,Saffola,GURGAON,DLHY GGNFC5,30,2026-09-30,Swiggy
3,54266,"Saffola Masala Oats Spicy Mexicana, Healthy & ...",MARICO LIMITED,Saffola,VIZAG,VIZ IM1,30,2026-09-30,Swiggy
4,55794,Parachute Advansed Baby Massage Oil (Virgin Co...,MARICO LIMITED,Parachute Advansed,GURGAON,DLHY GGNFC5,0,2026-09-30,Swiggy
...,...,...,...,...,...,...,...,...,...
10476,990631,"Beardo Beard & Hair Growth Oil, Natural Hair O...",MARICO LIMITED,Beardo,COIMBATORE,CBE ECOM,72,2026-09-30,Swiggy
10477,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,DELHI,DLHY GGNFC9,0,2026-09-30,Swiggy
10478,995855,Beardo UltraGlow Body Wash for Men Moisturizes...,MARICO LIMITED,Beardo,JAIPUR,JAI IM1,0,2026-09-30,Swiggy
10479,998784,Beardo LSD Lust Seduction Desire Eau De Parfum,MARICO LIMITED,Beardo,BANGALORE,BLR IM1,0,2026-09-30,Swiggy


In [57]:
blinkit_unpivoted = blinkit_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()
swiggy_unpivoted = Swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()


In [58]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
#chain_forecast_unpivoted = blinkit_unpivoted.copy()
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-09-30,262
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-09-30,150
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000364,2026-09-30,3526
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000365,2026-09-30,252
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000367,2026-09-30,3710
...,...,...,...,...,...
10476,Swiggy,VIZ IM1,978064,2026-09-30,24
10477,Swiggy,VIZ IM1,980876,2026-09-30,0
10478,Swiggy,VIZ IM1,987028,2026-09-30,20
10479,Swiggy,VIZ IM1,995855,2026-09-30,0


In [59]:
mapping = pd.read_excel("/data/aman_singh/mt_forecast/Daily Offtake Tracker - Jul'26.xlsb",sheet_name = 'Mapping')


In [60]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [61]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [62]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-09-30,262,Blinkit,10000059,8901088000772,718322,KL,5000.0
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-09-30,150,Blinkit,10000362,8901088002530,718328,KL,900.0
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000364,2026-09-30,3526,Blinkit,10000364,8901088034593,718398,KL,932.0
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000365,2026-09-30,252,Blinkit,10000365,8901088034616,718400,KL,4660.0
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000367,2026-09-30,3710,Blinkit,10000367,8901088068758,718560,TO,38.0
...,...,...,...,...,...,...,...,...,...,...,...
15998,Swiggy,VIZ IM1,978064,2026-09-30,24,NaN,NaN,NaN,NaN,NaN,NaN
15999,Swiggy,VIZ IM1,980876,2026-09-30,0,Swiggy,980876,8901088737067,732941,L,340.0
16000,Swiggy,VIZ IM1,987028,2026-09-30,20,Swiggy,987028,8901088732864,732011,L,50.0
16001,Swiggy,VIZ IM1,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name              0
facility_name           0
item_code               0
date                    0
forecast_quantity       0
platform_name        4091
asin                 4091
EAN                  4091
PSKU                 4091
UOM                  4091
Vol per unit         4091
dtype: int64

In [64]:
df_chk[(df_chk['PSKU'].isna()) & (df_chk['chain_name'] == 'Swiggy')]#['forecast_quantity'].sum()

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
5537,Swiggy,AHM DELHIVERY,4276,2026-09-30,60,NaN,NaN,NaN,NaN,NaN,NaN
5538,Swiggy,AHM DELHIVERY,4278,2026-09-30,432,NaN,NaN,NaN,NaN,NaN,NaN
5539,Swiggy,AHM DELHIVERY,4768,2026-09-30,96,NaN,NaN,NaN,NaN,NaN,NaN
5540,Swiggy,AHM DELHIVERY,4769,2026-09-30,144,NaN,NaN,NaN,NaN,NaN,NaN
5541,Swiggy,AHM DELHIVERY,4835,2026-09-30,112,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
15996,Swiggy,VIZ IM1,965003,2026-09-30,6,NaN,NaN,NaN,NaN,NaN,NaN
15997,Swiggy,VIZ IM1,977696,2026-09-30,24,NaN,NaN,NaN,NaN,NaN,NaN
15998,Swiggy,VIZ IM1,978064,2026-09-30,24,NaN,NaN,NaN,NaN,NaN,NaN
16001,Swiggy,VIZ IM1,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
df_chk[(df_chk['chain_name'] == 'Swiggy')]#['forecast_quantity'].sum()

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
5522,Swiggy,AHM DELHIVERY,3,2026-09-30,960,Swiggy,3,89002940,718299,KL,100.0
5523,Swiggy,AHM DELHIVERY,102,2026-09-30,180,Swiggy,102,8901088002530,718328,KL,900.0
5524,Swiggy,AHM DELHIVERY,103,2026-09-30,30,Swiggy,103,8901088047302,718449,L,190.0
5525,Swiggy,AHM DELHIVERY,104,2026-09-30,1120,Swiggy,104,8901088068741,718491,TO,38.0
5526,Swiggy,AHM DELHIVERY,105,2026-09-30,1960,Swiggy,105,8901088068758,718560,TO,38.0
...,...,...,...,...,...,...,...,...,...,...,...
15998,Swiggy,VIZ IM1,978064,2026-09-30,24,NaN,NaN,NaN,NaN,NaN,NaN
15999,Swiggy,VIZ IM1,980876,2026-09-30,0,Swiggy,980876,8901088737067,732941,L,340.0
16000,Swiggy,VIZ IM1,987028,2026-09-30,20,Swiggy,987028,8901088732864,732011,L,50.0
16001,Swiggy,VIZ IM1,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

166

In [67]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

In [68]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [69]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

In [70]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [71]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.642,1107
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-09-30,0.464,464
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718322,2026-09-30,1.310,262
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718328,2026-09-30,0.135,150
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718341,2026-09-30,1.889,1889
...,...,...,...,...,...,...
11804,Swiggy,VIZ IM1,810674,2026-09-30,1.008,72
11805,Swiggy,VIZ IM1,810685,2026-09-30,0.008,20
11806,Swiggy,VIZ IM1,810738,2026-09-30,0.000,0
11807,Swiggy,VIZ IM1,811181,2026-09-30,0.024,24


In [72]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0

In [73]:
facility_to_city_mappings_df = pd.read_excel(
    r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
    'Sheet1'
)
facility_to_city_mappings_df.columns = facility_to_city_mappings_df.columns.str.lower()
facility_to_city_mappings_df.columns = ['chain', 'facility_name', 'city', 'customer', 'marico_depot',
       'status', 'depot_name']
facility_to_city_mappings_df = facility_to_city_mappings_df[
    facility_to_city_mappings_df['customer'].notna()
]
facility_to_city_mappings_df['facility_name'] = facility_to_city_mappings_df['facility_name'].str.lower()
facility_to_city_mappings_df['city'] = facility_to_city_mappings_df['city'].str.lower()
facility_to_city_mappings_df.head()
customer_depot_mappings_df = pd.read_sql("""
SELECT DISTINCT customer_code, depot_code, channel_name 
FROM mst_customer
WHERE company_code='MIL' AND
    latest_record_ind=1
ORDER BY 3, 1, 2
""",
prod_conn
)
customer_depot_mappings_df.columns = customer_depot_mappings_df.columns.str.lower()
customer_depot_mappings_df.duplicated(subset=['customer_code']).sum()
customer_depot_mappings_df['customer_code'] = customer_depot_mappings_df['customer_code'].astype(str)
facility_to_city_mappings_df['customer'] = facility_to_city_mappings_df['customer'].astype(str)
customer_depot_mappings_df.dtypes
facility_to_city_mappings_df.dtypes
len_before_merge = len(facility_to_city_mappings_df)
facility_to_city_mappings_df = facility_to_city_mappings_df.merge(
    customer_depot_mappings_df[['customer_code', 'depot_code']].drop_duplicates().rename(
        columns={'customer_code': 'customer'}
    ),
    on=['customer'],
    how='left'
)
assert len_before_merge == len(facility_to_city_mappings_df)
del len_before_merge

In [74]:
facility_to_city_mappings_df#.isnull().sum()

,chain,facility_name,city,customer,marico_depot,status,depot_name,depot_code
0,Blinkit,farukhnagar f2 - feeder warehouse,gurugram,16012,D115,Active,NaN,D115
1,Blinkit,ahmedabad a2 - feeder warehouse,ahmedabad,15616,D354,Active,NaN,D354
2,Blinkit,hyderabad h3 - feeder warehouse,hyderabad,16005,D530,Active,NaN,D530
3,Blinkit,lucknow l5 - feeder warehouse,lucknow,15855,D113,Active,NaN,D113
4,Blinkit,super store hyderabad h2 - warehouse,hyderabad,9895,D530,Active,NaN,D530
...,...,...,...,...,...,...,...,...
144,Zepto,lko-dry-mh-sohramau,lucknow,16804,D113,Active,NaN,D113
145,Blinkit,hot mumbai m12 - feeder,mumbai,18603,D356,Active,NaN,D356
146,Blinkit,hot patna p2 - feeder,patna,18604,D233,Active,NaN,D233
147,Swiggy,scootsy logistics private limited- coimbatore,coimbatore,18366,D676,Active,NaN,D676


In [75]:
facility_to_city_mappings_df.rename(columns = {'facility_name':'FC', 'chain':'chain_name'}, inplace = True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()

facility_to_city_mappings_df[facility_to_city_mappings_df.duplicated(subset = ['chain_name','FC'],keep=False)]
facility_to_city_mappings_df = facility_to_city_mappings_df[['chain_name','FC','depot_code']].drop_duplicates()


In [76]:
df_chk.rename(columns = {'facility_name':'FC'},inplace = True)
df_chk['FC'] = df_chk['FC'].str.lower()
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,6.642,1107
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.464,464
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.310,262
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.135,150
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.889,1889
...,...,...,...,...,...,...
11804,Swiggy,viz im1,810674,2026-09-30,1.008,72
11805,Swiggy,viz im1,810685,2026-09-30,0.008,20
11806,Swiggy,viz im1,810738,2026-09-30,0.000,0
11807,Swiggy,viz im1,811181,2026-09-30,0.024,24


In [77]:
df_chk[df_chk['chain_name'] == 'Blinkit']['forecast_quantity'].sum()

2437306

In [78]:
chain_forecast_unpivoted[chain_forecast_unpivoted['chain_name'] == 'Blinkit']['forecast_quantity'].sum()

2437527

In [79]:
xy = df_chk.copy()

In [80]:
df_chk = df_chk.merge(facility_to_city_mappings_df, on = ['chain_name','FC'], how = 'left')
df_chk#.isnull().sum()

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,6.642,1107,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.464,464,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.310,262,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.135,150,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.889,1889,D354
...,...,...,...,...,...,...,...
11804,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572
11805,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572
11806,Swiggy,viz im1,810738,2026-09-30,0.000,0,D572
11807,Swiggy,viz im1,811181,2026-09-30,0.024,24,D572


In [81]:
df_chk[df_chk['depot_code'].isna()][['FC','chain_name']].drop_duplicates()#['vol_in_rum'].sum()/df_chk['vol_in_rum'].sum()

,FC,chain_name
1630,guwahati g2 - feeder warehouse,Blinkit
1861,indore i2 - feeder warehouse,Blinkit
2090,jaipur j4 - feeder warehouse,Blinkit
3219,mumbai m12 - feeder warehouse,Blinkit
3714,patna p2 - feeder warehouse,Blinkit
3956,pune p3 - feeder warehouse,Blinkit
4321,ranchi r2 - feeder warehouse,Blinkit
4806,surat s2 - feeder warehouse,Blinkit
5023,varanasi v2 - feeder warehouse,Blinkit
5130,vijayawada - feeder warehouse,Blinkit


In [82]:
df_chk[df_chk['depot_code'].isna()][['chain_name','FC']].drop_duplicates()

,chain_name,FC
1630,Blinkit,guwahati g2 - feeder warehouse
1861,Blinkit,indore i2 - feeder warehouse
2090,Blinkit,jaipur j4 - feeder warehouse
3219,Blinkit,mumbai m12 - feeder warehouse
3714,Blinkit,patna p2 - feeder warehouse
3956,Blinkit,pune p3 - feeder warehouse
4321,Blinkit,ranchi r2 - feeder warehouse
4806,Blinkit,surat s2 - feeder warehouse
5023,Blinkit,varanasi v2 - feeder warehouse
5130,Blinkit,vijayawada - feeder warehouse


In [83]:
df_chk[df_chk['chain_name'] == 'Swiggy']['vol_in_rum'].sum()

36714.444231999994

In [84]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,6.642,1107,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.464,464,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.310,262,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.135,150,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.889,1889,D354
...,...,...,...,...,...,...,...
11804,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572
11805,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572
11806,Swiggy,viz im1,810738,2026-09-30,0.000,0,D572
11807,Swiggy,viz im1,811181,2026-09-30,0.024,24,D572


In [85]:
material_master_df = pd.read_sql(
    """select * from mst_material 
    where latest_record_ind=1 and company_code='MIL'""",
    prod_conn
)
material_master_df.columns = material_master_df.columns.str.lower()
assert material_master_df.duplicated(
    subset=['company_code', 'material_code']).sum() == 0
material_master_df = material_master_df.rename(columns=
    {'material_group_code': 'brand_code'})
material_master_df['material_code'] = material_master_df['material_code'].astype(np.int64)
material_master_df.duplicated(subset=['material_code', 'parent_material_code', 'brand_code']).sum()

0

In [86]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)
len_before_merge = len(df_chk)
df_chk = df_chk.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(df_chk)

In [87]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,6.642,1107,D354,SAFF GOLD
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.464,464,D354,PCNO(R)
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.310,262,D354,SAFF KO
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.135,150,D354,SAFF KOCO
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.889,1889,D354,SAFF GOLD
...,...,...,...,...,...,...,...,...
11804,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572,PA_ESS_HO
11805,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572,SAF-MUSLI
11806,Swiggy,viz im1,810738,2026-09-30,0.000,0,D572,PABABY_GM
11807,Swiggy,viz im1,811181,2026-09-30,0.024,24,D572,SAF_CDPRS


In [88]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [89]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(df_chk)

df_chk = df_chk.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(df_chk)


Credentials retrieved successfully for prod db.


In [90]:
df_chk['value'] = df_chk['vol_in_rum']*df_chk['qtr_ind_rate']/10**7
df_chk[df_chk['chain_name'] == 'Swiggy'].groupby(['month_date'])['value'].sum()

month_date
2026-09-30    8.242319
Name: value, dtype: float64

In [91]:
df_chk[(df_chk['depot_code'].isna()) & (df_chk['chain_name'] == 'Swiggy')].groupby(['month_date'])['value'].sum()

month_date
2026-09-30    0.184551
Name: value, dtype: float64

In [92]:
# df_chk.to_csv('bl_sw_chain_fore.csv')

In [93]:
df_chk['value'].sum()

33.600787605504706

In [94]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_aug.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty
0,October,Ahmedabad,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0
1,September,Ahmedabad,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0
2,August,Ahmedabad,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0
3,September,Pune,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0
4,October,Pune,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6680,October,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,1758.0
6681,August,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,1536.0
6682,September,Bengaluru,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,736.0
6683,August,Bengaluru,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,736.0


In [95]:
chain_forecast_zepto['month'].unique()

array(['October', 'September', 'August'], dtype=object)

In [96]:

month_map = {
    'August': '2026-08-31',
    'September': '2026-09-30',
    'October': '2026-10-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_april = chain_forecast_zepto[chain_forecast_zepto['date'] == '2026-09-30']
chain_forecast_zepto_april

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
1,September,Ahmedabad,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0,2026-09-30
3,September,Pune,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0,2026-09-30
7,September,Ahmedabad,f4027831-5c6d-4847-a1de-3f4cc79a7f83,Just Herbs 12 Free Nail Paint-Blush Pink-05 | ...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,32.0,2026-09-30
11,September,Chennai,6a0638ab-3039-46d4-bf0e-72702cc0bbe8,Just Herbs 12 Free Nail Paint-Burgundy Beauty-...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,52.0,2026-09-30
13,September,Jaipur,6a0638ab-3039-46d4-bf0e-72702cc0bbe8,Just Herbs 12 Free Nail Paint-Burgundy Beauty-...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,4.0,2026-09-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6672,September,Bengaluru,d34d554d-671a-42a7-bd48-31301ccbf729,Just Herbs Pigmented Smudge & Sweat Proof Quic...,Makeup & Beauty,Face Makeup,Sindoor,Justherbs,Marico Limited,3.5,GRAM,225.0,64.0,2026-09-30
6675,September,Coimbatore,6096352f-c4f1-4f9b-9df2-80ec599177d6,Saffola Masala Oats | Curry & Pepper | Anytime...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,500.0,GRAM,226.0,30.0,2026-09-30
6676,September,Bengaluru,cee5fd8d-0637-4e6a-bcf5-6d97ebfd3681,Parachute Advansed Hair Fall Control Protein S...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,170.0,MILLILITRE,158.0,45.0,2026-09-30
6679,September,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,1724.0,2026-09-30


In [43]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_jun_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto
chain_forecast_zepto['month'].unique()

month_map = {
    'Jun': '2026-06-30',
    'May': '2026-05-31',
    'Jul': '2026-07-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_may = chain_forecast_zepto[chain_forecast_zepto['date'].isin(['2026-05-31'])]
chain_forecast_zepto_may

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
0,May,Jaipur,f1b1c703-46a6-43e2-8b49-43c6a193a2a2,Bio Oil Original Skincare Oil Suitable For Str...,Skincare,Body Care,Body Oil,Bio Oil,Marico Limited,60.0,MILLILITRE,550.0,20.0,2026-05-31
3,May,SAS Nagar,a74eaad2-e93b-46d3-8e6d-d11a56d38632,Parachute Advansed Baby Wipes with virgin coco...,Baby Care,Baby Wipes,Baby Wipes,Parachute,Marico Limited,1.0,PIECE,230.0,12.0,2026-05-31
8,May,Mumbai,a0841095-9583-4788-8c67-cbe41ddab0d7,Just Herbs Eyeliner | Multicolour | Waterproof,Makeup & Beauty,Eye Makeup,Eye Liner,Justherbs,Marico Limited,42.0,GRAM,599.0,60.0,2026-05-31
10,May,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,261.0,2104.0,2026-05-31
12,May,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,261.0,1104.0,2026-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,May,Jaipur,f702765e-ea47-41a5-90e8-3c797e6b29fe,Saffola 25g High Protein Oats | 14g Fibre | No...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,400.0,GRAM,299.0,28.0,2026-05-31
5581,May,Pune,03843645-8523-4571-8c92-7b3b5d292930,Livon Style Pro Keratin Serum 10X Stronger & S...,Hair Care,Hair Serum & Polish,Hair Serum,Livon,Marico Limited,100.0,MILLILITRE,665.0,12.0,2026-05-31
5583,May,Kolkata,da954c80-83a5-43f0-8c50-2354cd689945,Just Herbs Hair Growth Oil With Rosemary And C...,Hair Care,Hair Oil,Hair Growth Oil,Justherbs,Marico Limited,100.0,MILLILITRE,345.0,8.0,2026-05-31
5586,May,Pune,d34d554d-671a-42a7-bd48-31301ccbf729,Just Herbs Pigmented Smudge & Sweat Proof Quic...,Makeup & Beauty,Face Makeup,Sindoor,Justherbs,Marico Limited,3.5,GRAM,225.0,49.0,2026-05-31


In [44]:
chain_forecast_zepto_may['projected_qty'].sum()

1280872.0

In [13]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_may_zepto.csv')
chain_forecast_zepto['Month'].unique()

array(['Jul', 'Aug', 'Jun'], dtype=object)

In [15]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_may_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto
chain_forecast_zepto['month'].unique()

month_map = {
    'Jun': '2026-06-30',
    'Aug': '2026-08-31',
    'Jul': '2026-07-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_june = chain_forecast_zepto[chain_forecast_zepto['date'].isin(['2026-06-30'])]
chain_forecast_zepto_june

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
4,Jun,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,2278,2026-06-30
6,Jun,Bengaluru,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,4991,2026-06-30
8,Jun,Pune,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1389,2026-06-30
10,Jun,Hyderabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,7467,2026-06-30
16,Jun,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1069,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Jun,Kolkata,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,1064,2026-06-30
534,Jun,Kolkata,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,950,2026-06-30
536,Jun,Mumbai,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,171,2026-06-30
541,Jun,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,1386,2026-06-30


In [97]:
# chain_forecast_zepto = pd.concat([chain_forecast_zepto_april,chain_forecast_zepto_may,chain_forecast_zepto_june])
chain_forecast_zepto = chain_forecast_zepto_april.copy()
chain_forecast_zepto

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
1,September,Ahmedabad,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0,2026-09-30
3,September,Pune,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0,2026-09-30
7,September,Ahmedabad,f4027831-5c6d-4847-a1de-3f4cc79a7f83,Just Herbs 12 Free Nail Paint-Blush Pink-05 | ...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,32.0,2026-09-30
11,September,Chennai,6a0638ab-3039-46d4-bf0e-72702cc0bbe8,Just Herbs 12 Free Nail Paint-Burgundy Beauty-...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,52.0,2026-09-30
13,September,Jaipur,6a0638ab-3039-46d4-bf0e-72702cc0bbe8,Just Herbs 12 Free Nail Paint-Burgundy Beauty-...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,4.0,2026-09-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6672,September,Bengaluru,d34d554d-671a-42a7-bd48-31301ccbf729,Just Herbs Pigmented Smudge & Sweat Proof Quic...,Makeup & Beauty,Face Makeup,Sindoor,Justherbs,Marico Limited,3.5,GRAM,225.0,64.0,2026-09-30
6675,September,Coimbatore,6096352f-c4f1-4f9b-9df2-80ec599177d6,Saffola Masala Oats | Curry & Pepper | Anytime...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,500.0,GRAM,226.0,30.0,2026-09-30
6676,September,Bengaluru,cee5fd8d-0637-4e6a-bcf5-6d97ebfd3681,Parachute Advansed Hair Fall Control Protein S...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,170.0,MILLILITRE,158.0,45.0,2026-09-30
6679,September,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,1724.0,2026-09-30


In [98]:
chain_forecast_zepto['chain_name'] = 'Zepto'
chain_forecast_zepto.rename(columns = {'product_variant_id':'item_code', 
                                       'projected_qty':'forecast_quantity',
                                       'cluster_dry':'city'}, inplace = True)
chain_forecast_zepto

,month,city,item_code,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,forecast_quantity,date,chain_name
1,September,Ahmedabad,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0,2026-09-30,Zepto
3,September,Pune,fd3c932f-93e2-4de7-b372-6c699f0b97bb,Parachute Advansed Damage Repair Protein Shamp...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,650.0,MILLILITRE,700.0,4.0,2026-09-30,Zepto
7,September,Ahmedabad,f4027831-5c6d-4847-a1de-3f4cc79a7f83,Just Herbs 12 Free Nail Paint-Blush Pink-05 | ...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,32.0,2026-09-30,Zepto
11,September,Chennai,6a0638ab-3039-46d4-bf0e-72702cc0bbe8,Just Herbs 12 Free Nail Paint-Burgundy Beauty-...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,52.0,2026-09-30,Zepto
13,September,Jaipur,6a0638ab-3039-46d4-bf0e-72702cc0bbe8,Just Herbs 12 Free Nail Paint-Burgundy Beauty-...,Makeup & Beauty,Nails,Nail Polish,Justherbs,Marico Limited,11.0,MILLILITRE,149.0,4.0,2026-09-30,Zepto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6672,September,Bengaluru,d34d554d-671a-42a7-bd48-31301ccbf729,Just Herbs Pigmented Smudge & Sweat Proof Quic...,Makeup & Beauty,Face Makeup,Sindoor,Justherbs,Marico Limited,3.5,GRAM,225.0,64.0,2026-09-30,Zepto
6675,September,Coimbatore,6096352f-c4f1-4f9b-9df2-80ec599177d6,Saffola Masala Oats | Curry & Pepper | Anytime...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,500.0,GRAM,226.0,30.0,2026-09-30,Zepto
6676,September,Bengaluru,cee5fd8d-0637-4e6a-bcf5-6d97ebfd3681,Parachute Advansed Hair Fall Control Protein S...,Hair Care,Shampoo,Shampoo,Parachute,Marico Limited,170.0,MILLILITRE,158.0,45.0,2026-09-30,Zepto
6679,September,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,114.0,1724.0,2026-09-30,Zepto


In [99]:
chain_forecast_zepto.isnull().sum()

month                0
city                 0
item_code            0
product_name         0
category_name        0
subcategory_name     0
l3_category_name     0
brand_name           0
manufacturer         0
packsize             0
unit_of_measure      0
unit_mrp             0
forecast_quantity    0
date                 0
chain_name           0
dtype: int64

In [100]:
chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','city','item_code','date'])['forecast_quantity'].sum().reset_index()
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity
0,Zepto,Ahmedabad,00846fd7-7624-469f-8df3-ef6b3279ede8,2026-09-30,8.0
1,Zepto,Ahmedabad,00f3f58c-c7f7-4b55-a374-e5469ad19223,2026-09-30,350.0
2,Zepto,Ahmedabad,01400883-9518-4f73-9c9f-593130e0d414,2026-09-30,156.0
3,Zepto,Ahmedabad,021c7b96-7a79-492b-9226-f74c9e5ce670,2026-09-30,32.0
4,Zepto,Ahmedabad,0273ef06-8fa6-49e5-8b69-715d2bc3e60b,2026-09-30,248.0
...,...,...,...,...,...
2221,Zepto,SAS Nagar,f9bc0f24-4ff3-4b04-80e1-3e91c1931acf,2026-09-30,12.0
2222,Zepto,SAS Nagar,f9dacf29-3a8f-4d9e-90c2-f2acb4dbd253,2026-09-30,8.0
2223,Zepto,SAS Nagar,fa1b347b-9200-410a-831b-4471c4b2c3c8,2026-09-30,168.0
2224,Zepto,SAS Nagar,fc0bdd04-2f61-4a5c-9126-486b4c041a76,2026-09-30,176.0


In [101]:
zepto_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'Zepto')
zepto_mapping = zepto_mapping[zepto_mapping['Status'] == 'Active']
zepto_mapping = zepto_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
zepto_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
zepto_mapping['platform_name'] = zepto_mapping['platform_name'].str.lower()
zepto_mapping['city'] = zepto_mapping['city'].str.lower()
zepto_mapping

,platform_name,city,depot
0,zepto,ahmedabad,D354
2,zepto,indore,D354
3,zepto,mehsana,D354
4,zepto,rajkot,D354
5,zepto,surat,D354
...,...,...,...
110,zepto,pune,D461
111,zepto,bahadurgarh,D115
112,zepto,gurgaon,NaN
113,zepto,raipur,D465


In [102]:
zepto_mapping[zepto_mapping['city'] == 'NCR']

,platform_name,city,depot


In [103]:
xx = chain_forecast_zepto.copy()

In [104]:
chain_forecast_zepto['city'] = chain_forecast_zepto['city'].str.lower()
chain_forecast_zepto = chain_forecast_zepto.merge(zepto_mapping, on = ['city'], how = 'left')
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name,depot
0,Zepto,ahmedabad,00846fd7-7624-469f-8df3-ef6b3279ede8,2026-09-30,8.0,zepto,D354
1,Zepto,ahmedabad,00f3f58c-c7f7-4b55-a374-e5469ad19223,2026-09-30,350.0,zepto,D354
2,Zepto,ahmedabad,01400883-9518-4f73-9c9f-593130e0d414,2026-09-30,156.0,zepto,D354
3,Zepto,ahmedabad,021c7b96-7a79-492b-9226-f74c9e5ce670,2026-09-30,32.0,zepto,D354
4,Zepto,ahmedabad,0273ef06-8fa6-49e5-8b69-715d2bc3e60b,2026-09-30,248.0,zepto,D354
...,...,...,...,...,...,...,...
2221,Zepto,sas nagar,f9bc0f24-4ff3-4b04-80e1-3e91c1931acf,2026-09-30,12.0,zepto,D115
2222,Zepto,sas nagar,f9dacf29-3a8f-4d9e-90c2-f2acb4dbd253,2026-09-30,8.0,zepto,D115
2223,Zepto,sas nagar,fa1b347b-9200-410a-831b-4471c4b2c3c8,2026-09-30,168.0,zepto,D115
2224,Zepto,sas nagar,fc0bdd04-2f61-4a5c-9126-486b4c041a76,2026-09-30,176.0,zepto,D115


In [105]:
chain_forecast_zepto[chain_forecast_zepto['depot'].isna()]#['city'].unique()

,chain_name,city,item_code,date,forecast_quantity,platform_name,depot


In [106]:
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto['item_code'] = chain_forecast_zepto['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [107]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto = chain_forecast_zepto.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(chain_forecast_zepto))
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,00846fd7-7624-469f-8df3-ef6b3279ede8,2026-09-30,8.0,zepto,D354,NaN,NaN,NaN,NaN,NaN,NaN
1,Zepto,ahmedabad,00f3f58c-c7f7-4b55-a374-e5469ad19223,2026-09-30,350.0,zepto,D354,Zepto,00f3f58c-c7f7-4b55-a374-e5469ad19223,8901088766951,809818,TO,185.0
2,Zepto,ahmedabad,01400883-9518-4f73-9c9f-593130e0d414,2026-09-30,156.0,zepto,D354,Zepto,01400883-9518-4f73-9c9f-593130e0d414,8901088069113,808731,TO,500.0
3,Zepto,ahmedabad,021c7b96-7a79-492b-9226-f74c9e5ce670,2026-09-30,32.0,zepto,D354,Zepto,021c7b96-7a79-492b-9226-f74c9e5ce670,8901088783460,728768,L,300.0
4,Zepto,ahmedabad,0273ef06-8fa6-49e5-8b69-715d2bc3e60b,2026-09-30,248.0,zepto,D354,Zepto,0273ef06-8fa6-49e5-8b69-715d2bc3e60b,8901088158831,719084,L,400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2221,Zepto,sas nagar,f9bc0f24-4ff3-4b04-80e1-3e91c1931acf,2026-09-30,12.0,zepto,D115,NaN,NaN,NaN,NaN,NaN,NaN
2222,Zepto,sas nagar,f9dacf29-3a8f-4d9e-90c2-f2acb4dbd253,2026-09-30,8.0,zepto,D115,Zepto,f9dacf29-3a8f-4d9e-90c2-f2acb4dbd253,8901088703239,730712,L,100.0
2223,Zepto,sas nagar,fa1b347b-9200-410a-831b-4471c4b2c3c8,2026-09-30,168.0,zepto,D115,Zepto,fa1b347b-9200-410a-831b-4471c4b2c3c8,8901088138130,718836,KG,60.0
2224,Zepto,sas nagar,fc0bdd04-2f61-4a5c-9126-486b4c041a76,2026-09-30,176.0,zepto,D115,Zepto,fc0bdd04-2f61-4a5c-9126-486b4c041a76,8901088210409,721195,L,250.0


In [57]:
xx = chain_forecast_zepto.copy()

In [108]:
chain_forecast_zepto.isnull().sum()

chain_name             0
city                   0
item_code              0
date                   0
forecast_quantity      0
platform_name_x        0
depot                  0
platform_name_y      958
asin                 958
EAN                  958
PSKU                 958
UOM                  958
Vol per unit         958
dtype: int64

In [109]:
chain_forecast_zepto.dropna(subset = ['PSKU'],inplace = True)
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
1,Zepto,ahmedabad,00f3f58c-c7f7-4b55-a374-e5469ad19223,2026-09-30,350.0,zepto,D354,Zepto,00f3f58c-c7f7-4b55-a374-e5469ad19223,8901088766951,809818,TO,185.0
2,Zepto,ahmedabad,01400883-9518-4f73-9c9f-593130e0d414,2026-09-30,156.0,zepto,D354,Zepto,01400883-9518-4f73-9c9f-593130e0d414,8901088069113,808731,TO,500.0
3,Zepto,ahmedabad,021c7b96-7a79-492b-9226-f74c9e5ce670,2026-09-30,32.0,zepto,D354,Zepto,021c7b96-7a79-492b-9226-f74c9e5ce670,8901088783460,728768,L,300.0
4,Zepto,ahmedabad,0273ef06-8fa6-49e5-8b69-715d2bc3e60b,2026-09-30,248.0,zepto,D354,Zepto,0273ef06-8fa6-49e5-8b69-715d2bc3e60b,8901088158831,719084,L,400.0
5,Zepto,ahmedabad,03843645-8523-4571-8c92-7b3b5d292930,2026-09-30,24.0,zepto,D354,Zepto,03843645-8523-4571-8c92-7b3b5d292930,8901088891684,809250,KL,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2219,Zepto,sas nagar,f881826f-130e-495b-a6cb-03f089aaa68f,2026-09-30,8.0,zepto,D115,Zepto,f881826f-130e-495b-a6cb-03f089aaa68f,8901088783422,728767,L,300.0
2220,Zepto,sas nagar,f9435600-2d79-477d-942d-19d0c7307403,2026-09-30,104.0,zepto,D115,Zepto,f9435600-2d79-477d-942d-19d0c7307403,8901088151801,731857,L,500.0
2222,Zepto,sas nagar,f9dacf29-3a8f-4d9e-90c2-f2acb4dbd253,2026-09-30,8.0,zepto,D115,Zepto,f9dacf29-3a8f-4d9e-90c2-f2acb4dbd253,8901088703239,730712,L,100.0
2223,Zepto,sas nagar,fa1b347b-9200-410a-831b-4471c4b2c3c8,2026-09-30,168.0,zepto,D115,Zepto,fa1b347b-9200-410a-831b-4471c4b2c3c8,8901088138130,718836,KG,60.0


In [ ]:
# chain_forecast_zepto.duplicated(subset=['chain_name','PSKU','date'], keep=False).sum()

2960

In [110]:
chain_forecast_zepto['month_date'] = chain_forecast_zepto['date'] + pd.offsets.MonthEnd(0)

chain_forecast_zepto.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
chain_forecast_zepto['vol_in_lit'] = chain_forecast_zepto['forecast_quantity']*chain_forecast_zepto['vol_per_unit']/1000
chain_forecast_zepto['vol_in_rum'] = chain_forecast_zepto.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','depot','PSKU','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
chain_forecast_zepto['PSKU'] = chain_forecast_zepto['PSKU'].astype(int)
chain_forecast_zepto.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
chain_forecast_zepto

,chain_name,depot,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Zepto,D112,718288,2026-09-30,18.030,3005.0
1,Zepto,D112,718312,2026-09-30,1.705,1705.0
2,Zepto,D112,718322,2026-09-30,5.220,1044.0
3,Zepto,D112,718341,2026-09-30,13.584,13584.0
4,Zepto,D112,718342,2026-09-30,3.336,1668.0
...,...,...,...,...,...,...
1170,Zepto,D674,810518,2026-09-30,0.216,216.0
1171,Zepto,D674,810519,2026-09-30,0.184,184.0
1172,Zepto,D674,810673,2026-09-30,0.336,24.0
1173,Zepto,D674,810674,2026-09-30,0.336,24.0


In [111]:
chain_forecast_zepto['forecast_quantity'].sum()

1307222.0

In [62]:
xx['forecast_quantity'].sum()

1352103.0

In [77]:
chain_forecast_zepto.to_csv('forecast_zepto.csv')

In [57]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-08-31,5.2080,D354,SAFF GOLD,138865.260689,0.072321
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-08-31,0.3970,D354,PCNO(R),349274.001420,0.013866
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-08-31,0.9450,D354,SAFF KO,168827.536176,0.015954
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-08-31,0.1152,D354,SAFF KOCO,123636.889888,0.001424
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-08-31,1.6580,D354,SAFF GOLD,138865.260689,0.023024
...,...,...,...,...,...,...,...,...,...
11721,Swiggy,viz im1,810522,2026-08-31,0.0240,D572,SAF_CDPRS,260000.000000,0.000624
11722,Swiggy,viz im1,810673,2026-08-31,0.1120,D572,PA_ESS_HO,12860.631072,0.000144
11723,Swiggy,viz im1,810674,2026-08-31,1.0080,D572,PA_ESS_HO,12860.631072,0.001296
11724,Swiggy,viz im1,810685,2026-08-31,0.0080,D572,SAF-MUSLI,315513.490535,0.000252


In [112]:
fc_depot_mapping = {
    'guwahati g2': 'D236',
    'indore i2': 'D464',
    'jaipur j4': 'D314',
    'mumbai m12': 'D356',
    'patna p2': 'D233',
    'pune p3': 'D461',
    'ranchi r2': 'D234',
    'surat s2': 'D354',
    'varanasi v2': 'D113',
    'vijayawada': 'D572',
    'blr im4': 'D673'
}

# Fill only null depot codes
df_chk.loc[df_chk['depot_code'].isna(), 'depot_code'] = (
    df_chk.loc[df_chk['depot_code'].isna(), 'FC']
          .str.lower()
          .map(fc_depot_mapping)
)
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,6.642,1107,D354,SAFF GOLD,138865.260689,0.092234
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.464,464,D354,PCNO(R),349274.001420,0.016206
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.310,262,D354,SAFF KO,168827.536176,0.022116
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.135,150,D354,SAFF KOCO,123636.889888,0.001669
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.889,1889,D354,SAFF GOLD,138865.260689,0.026232
...,...,...,...,...,...,...,...,...,...,...
11804,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572,PA_ESS_HO,12860.631072,0.001296
11805,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572,SAF-MUSLI,315513.490535,0.000252
11806,Swiggy,viz im1,810738,2026-09-30,0.000,0,D572,PABABY_GM,366.484998,0.000000
11807,Swiggy,viz im1,811181,2026-09-30,0.024,24,D572,SAF_CDPRS,260000.000000,0.000624


In [113]:
df_chk = df_chk.groupby(['chain_name','depot_code','parent_material_code','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
df_chk

,chain_name,depot_code,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-09-30,0.0420,7
1,Blinkit,D112,718312,2026-09-30,1.7380,1738
2,Blinkit,D112,718322,2026-09-30,2.7350,547
3,Blinkit,D112,718328,2026-09-30,0.2619,291
4,Blinkit,D112,718330,2026-09-30,0.0200,4
...,...,...,...,...,...,...
6546,Swiggy,D677,810673,2026-09-30,0.5040,36
6547,Swiggy,D677,810674,2026-09-30,0.5040,36
6548,Swiggy,D677,810738,2026-09-30,8.6880,24
6549,Swiggy,D677,811181,2026-09-30,0.0120,12


In [114]:
df_chk.rename(columns = {'depot_code':'depot'},inplace = True)
final_df = pd.concat([df_chk,chain_forecast_zepto])
final_df

,chain_name,depot,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-09-30,0.0420,7.0
1,Blinkit,D112,718312,2026-09-30,1.7380,1738.0
2,Blinkit,D112,718322,2026-09-30,2.7350,547.0
3,Blinkit,D112,718328,2026-09-30,0.2619,291.0
4,Blinkit,D112,718330,2026-09-30,0.0200,4.0
...,...,...,...,...,...,...
1170,Zepto,D674,810518,2026-09-30,0.2160,216.0
1171,Zepto,D674,810519,2026-09-30,0.1840,184.0
1172,Zepto,D674,810673,2026-09-30,0.3360,24.0
1173,Zepto,D674,810674,2026-09-30,0.3360,24.0


In [115]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()
realignment_df['channel'].unique()
realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'All'])]
def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data


In [116]:
final_df.columns

Index(['chain_name', 'depot', 'parent_material_code', 'month_date',
       'vol_in_rum', 'forecast_quantity'],
      dtype='object')

In [117]:
final_df['realigned_psku'] = final_df['parent_material_code'].copy()
final_df = realign_pskus(final_df, 'realigned_psku')
base_df = final_df.copy()
final_df = final_df.groupby(
    ['chain_name', 'depot', 'realigned_psku', 'month_date'],
    as_index=False
)[[ 'vol_in_rum', 'forecast_quantity']].sum()
final_df.duplicated(
    subset=['chain_name', 'depot', 'realigned_psku', 'month_date']).sum()

0

In [ ]:
# final_df.isnull().sum()

chain_name           0
depot                0
realigned_psku       0
month_date           0
vol_in_rum           0
forecast_quantity    0
dtype: int64

In [118]:
xx = final_df.copy()

In [102]:
xx

,chain_name,depot,realigned_psku,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-08-31,0.0420,7.0
1,Blinkit,D112,718312,2026-08-31,1.7480,1748.0
2,Blinkit,D112,718322,2026-08-31,2.6650,533.0
3,Blinkit,D112,718328,2026-08-31,0.2574,286.0
4,Blinkit,D112,718330,2026-08-31,0.0200,4.0
...,...,...,...,...,...,...
7633,Zepto,D674,810439,2026-08-31,0.2828,404.0
7634,Zepto,D674,810518,2026-08-31,0.2160,216.0
7635,Zepto,D674,810673,2026-08-31,0.3360,24.0
7636,Zepto,D674,810674,2026-08-31,0.3360,24.0


In [78]:
final_df.to_csv('qcom_chain_forecast_aug2.csv')

In [119]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)


In [120]:
material_master_df[['parent_material_code', 'brand_code']].drop_duplicates().duplicated(subset=[ 'parent_material_code']).sum()

0

In [121]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['realigned_psku'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(final_df)

In [122]:
final_df

,chain_name,depot,realigned_psku,month_date,vol_in_rum,forecast_quantity,parent_material_code,brand_code
0,Blinkit,D112,718288,2026-09-30,0.0420,7.0,718288,SAFF GOLD
1,Blinkit,D112,718312,2026-09-30,1.7380,1738.0,718312,PCNO(R)
2,Blinkit,D112,718322,2026-09-30,2.7350,547.0,718322,SAFF KO
3,Blinkit,D112,718328,2026-09-30,0.2619,291.0,718328,SAFF KOCO
4,Blinkit,D112,718330,2026-09-30,0.0200,4.0,718330,SAFF KOCO
...,...,...,...,...,...,...,...,...
7681,Zepto,D674,810439,2026-09-30,0.2828,404.0,810439,SAF-MUSLI
7682,Zepto,D674,810518,2026-09-30,0.2160,216.0,810518,SAF_CDPRS
7683,Zepto,D674,810673,2026-09-30,0.3360,24.0,810673,PA_ESS_HO
7684,Zepto,D674,810674,2026-09-30,0.3360,24.0,810674,PA_ESS_HO


In [ ]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-08-31,5.2080,868,D354,SAFF GOLD
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-08-31,0.3970,397,D354,PCNO(R)
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-08-31,0.9450,189,D354,SAFF KO
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-08-31,0.1152,128,D354,SAFF KOCO
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-08-31,1.6580,1658,D354,SAFF GOLD
...,...,...,...,...,...,...,...,...
11721,Swiggy,viz im1,810522,2026-08-31,0.0240,24,D572,SAF_CDPRS
11722,Swiggy,viz im1,810673,2026-08-31,0.1120,8,D572,PA_ESS_HO
11723,Swiggy,viz im1,810674,2026-08-31,1.0080,72,D572,PA_ESS_HO
11724,Swiggy,viz im1,810685,2026-08-31,0.0080,20,D572,SAF-MUSLI


In [123]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [124]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [125]:
final_df['value'] = final_df['vol_in_rum']*final_df['qtr_ind_rate']/10**7
final_df[final_df['chain_name'] == 'Swiggy'].groupby(['month_date'])['value'].sum()

month_date
2026-09-30    8.232793
Name: value, dtype: float64

In [127]:
final_df

,chain_name,depot,realigned_psku,month_date,vol_in_rum,forecast_quantity,parent_material_code,brand_code,qtr_ind_rate,value
0,Blinkit,D112,718288,2026-09-30,0.0420,7.0,718288,SAFF GOLD,138865.260689,0.000583
1,Blinkit,D112,718312,2026-09-30,1.7380,1738.0,718312,PCNO(R),349274.001420,0.060704
2,Blinkit,D112,718322,2026-09-30,2.7350,547.0,718322,SAFF KO,168827.536176,0.046174
3,Blinkit,D112,718328,2026-09-30,0.2619,291.0,718328,SAFF KOCO,123636.889888,0.003238
4,Blinkit,D112,718330,2026-09-30,0.0200,4.0,718330,SAFF KOCO,123636.889888,0.000247
...,...,...,...,...,...,...,...,...,...,...
7681,Zepto,D674,810439,2026-09-30,0.2828,404.0,810439,SAF-MUSLI,315513.490535,0.008923
7682,Zepto,D674,810518,2026-09-30,0.2160,216.0,810518,SAF_CDPRS,260000.000000,0.005616
7683,Zepto,D674,810673,2026-09-30,0.3360,24.0,810673,PA_ESS_HO,12860.631072,0.000432
7684,Zepto,D674,810674,2026-09-30,0.3360,24.0,810674,PA_ESS_HO,12860.631072,0.000432


In [126]:
final_df.to_csv('qcom_chain_forecast_sep.csv')

### The end

In [71]:
primary = pd.read_csv('/data/aman_singh/acuuracy_check/QCOM Chain PSKU OTP Output/live_runs/QCOM Chain Depot PSKU Primary_live_runs_06_Jul_2026.csv')
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
primary.columns

Index(['Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain depot PSKU Lag 3 Val',
       'Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU Actuals Val',
       'LY Offtake Chain depot PSKU Lag 1 Val',
       'LY Offtake Chain depot PSKU Lag 2 Val',
       'LY Offtake Chain depot PSKU Lag 3 Val',
       'LY Offtake Chain depot PSKU Lead 1 Val',
       'LY Offtake Chain depot PSKU Lead 2 Val', 'Calculated Primary Val'],
      dtype='object', length=117)

In [73]:
final_df.columns = ['Chain', 'Depot', 'PSKU','Month Date','Chain_primary_vol']
final_df['Depot'] = final_df['Depot'].str.lower()
final_df

,Chain,Depot,PSKU,Month Date,Chain_primary_vol
0,Blinkit,d112,718288,2026-08-31,0.0420
1,Blinkit,d112,718312,2026-08-31,1.9460
2,Blinkit,d112,718322,2026-08-31,3.5400
3,Blinkit,d112,718328,2026-08-31,0.2151
4,Blinkit,d112,718330,2026-08-31,0.0250
...,...,...,...,...,...
1245,Zepto,d674,810518,2026-08-31,0.3080
1246,Zepto,d674,810519,2026-08-31,0.1000
1247,Zepto,d674,810673,2026-08-31,0.5040
1248,Zepto,d674,810674,2026-08-31,0.3360


In [74]:
primary['Month Date'] = pd.to_datetime(primary['Month Date'])
primary = primary.merge(final_df, on = ['Chain', 'Depot', 'PSKU','Month Date'], how = 'left')
primary['Chain_primary_val'] = primary['Chain_primary_vol']*primary['Index Rate']/10**7
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val,Chain_primary_vol,Chain_primary_val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [75]:
primary[primary['Month Date']=='2026-08-31']['Chain_primary_val'].sum()

41.958339936520986

In [76]:
primary.to_csv('cdp_c_forecast.csv')

In [66]:
x['Chain_primary_vol'].isnull().sum()

207123

In [67]:
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()

,chain_name,facility_name,item_code,date,forecast_quantity
0,Swiggy,AHM DELHIVERY,3,2026-07-31,192
1,Swiggy,AHM DELHIVERY,3,2026-08-31,768
2,Swiggy,AHM DELHIVERY,3,2026-09-30,576
3,Swiggy,AHM DELHIVERY,102,2026-07-31,60
4,Swiggy,AHM DELHIVERY,102,2026-08-31,160
...,...,...,...,...,...
29878,Swiggy,VIZ IM1,995855,2026-08-31,0
29879,Swiggy,VIZ IM1,995855,2026-09-30,0
29880,Swiggy,VIZ IM1,999977,2026-07-31,4
29881,Swiggy,VIZ IM1,999977,2026-08-31,5


In [ ]:
blinkit_unpivoted = blinkit_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]
swiggy_unpivoted = swiggy_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]


In [ ]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65
...,...,...,...,...,...
29878,Swiggy,PUN DELHIVERY,990631,2026-09-30,22
29879,Swiggy,CHD ECOM,991861,2026-09-30,24
29880,Swiggy,CHN ECOM,995855,2026-09-30,0
29881,Swiggy,HYD IM1,998784,2026-09-30,0


In [ ]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [ ]:
# # Keys to match rows on
# keys = ["platform_name", "asin", "EAN", "PSKU", "UOM", "Vol per unit"]

# # Build a small DataFrame with the rows to drop
# rows_to_drop = pd.DataFrame([
#     {
#         "platform_name": "Zepto",
#         "asin": "0523a4ba-32cf-4e59-abd8-0e4086859b39",
#         "EAN": "8901088205924",
#         "PSKU": "718729",
#         "UOM": "L",
#         "Vol per unit": 100.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "197827dc-3184-4c57-a966-5461967bcb3a",
#         "EAN": "8901088884402",
#         "PSKU": "808485",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8906051370753",
#         "PSKU": "807069",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8901088075817",
#         "PSKU": "808262",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Swiggy",
#         "asin": "944906",
#         "EAN": "8901088150095",
#         "PSKU": "718976",
#         "UOM": "L",
#         "Vol per unit": 300.0,
#     },
# ])

# # Mark rows to drop via left-merge on keys
# _marked = temp.merge(
#     rows_to_drop.assign(_drop=1),
#     on=keys,
#     how="left",
#     validate="m:m"  # remove if unsure about duplicates
# )

# # Keep everything that was not marked to drop
# temp_clean = _marked[_marked["_drop"].isna()].drop(columns=["_drop"])
# temp_clean
# duplicates = temp_clean[temp_clean.duplicated(subset="asin", keep=False)]
# duplicates

In [ ]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [ ]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856,Blinkit,10171388,8901088213608,721427,TO,1000.0
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18,Blinkit,10232351,8901088796804,810520,KL,1000.0
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12,Blinkit,10029462,6001159111856,807033,L,125.0
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209,Blinkit,10015827,8901088043953,718312,KL,1000.0
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65,Blinkit,10116052,8901088205993,721133,L,300.0
...,...,...,...,...,...,...,...,...,...,...,...
50086,Swiggy,PUN DELHIVERY,990631,2026-09-30,22,NaN,NaN,NaN,NaN,NaN,NaN
50087,Swiggy,CHD ECOM,991861,2026-09-30,24,Swiggy,991861,8906027074531,729893,L,240.0
50088,Swiggy,CHN ECOM,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN
50089,Swiggy,HYD IM1,998784,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
date                     0
forecast_quantity        0
platform_name        12111
asin                 12111
EAN                  12111
PSKU                 12111
UOM                  12111
Vol per unit         12111
dtype: int64

In [ ]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

474

In [ ]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
20560,Swiggy,AHM DELHIVERY,554793,2026-07-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
23718,Swiggy,AHM DELHIVERY,60103,2026-07-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
30521,Swiggy,AHM DELHIVERY,554793,2026-08-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
33679,Swiggy,AHM DELHIVERY,60103,2026-08-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
40482,Swiggy,AHM DELHIVERY,554793,2026-09-30,0,Swiggy,554793,8901088886970,809042,KG,225.0
43640,Swiggy,AHM DELHIVERY,60103,2026-09-30,120,Swiggy,60103,8901088886970,809042,KG,225.0
21827,Swiggy,BLR DHL,819548,2026-07-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
23859,Swiggy,BLR DHL,298412,2026-07-31,0,Swiggy,298412,8901088171755,719162,TO,250.0
31788,Swiggy,BLR DHL,819548,2026-08-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
33820,Swiggy,BLR DHL,298412,2026-08-31,0,Swiggy,298412,8901088171755,719162,TO,250.0


In [ ]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [ ]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

0

In [ ]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [ ]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-07-31,5.754,959
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-08-31,6.204,1034
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.222,1037
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-10-31,8.568,1428
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-07-31,0.501,501
...,...,...,...,...,...,...
37674,Swiggy,VIZ IM1,810685,2026-08-31,0.008,20
37675,Swiggy,VIZ IM1,810685,2026-09-30,0.008,20
37676,Swiggy,VIZ IM1,810738,2026-07-31,0.000,0
37677,Swiggy,VIZ IM1,810738,2026-08-31,0.000,0


In [ ]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0